Import Modules, define network and apply transformations

In [1]:
import torch
from torchvision.datasets import Flowers102
import scipy
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

class FlowersNetwork(nn.Module):
  def __init__(self, in_feature, hidden_layers, out_features, activation_function = F.relu):
    super().__init__()

    self.layers = []
    self.layers.append(nn.Linear(in_feature, hidden_layers[0]))
    self.add_module("inputLayer", self.layers[0])

    for i in range(1, len(hidden_layers)):
      self.layers.append(nn.Linear(hidden_layers[i-1], hidden_layers[i]))
      self.add_module(f"hiddenLayer{i}", self.layers[i])

    self.out = nn.Linear(hidden_layers[-1], out_features)

    self.activation_function = activation_function

  def forward(self, x):
    for i in range(len(self.layers)):
      x = self.activation_function(self.layers[i](x))
    x = self.out(x)
    return x

# Set image dimensions
imageWidth = 224
imageHeight = 224

# Define batch size
batch_size = 32

# Calculate mean and standard deviation of the dataset for normalisation
flowersTransform_no_norm = transforms.Compose([
    transforms.Resize((imageWidth, imageHeight)),
    transforms.ToTensor()
])

flowersTrain_no_norm = Flowers102(root="./data", split="train", download=True, transform=flowersTransform_no_norm)
train_no_norm_loader = DataLoader(flowersTrain_no_norm, batch_size=batch_size, shuffle=False)

mean = 0.
std = 0.
total_samples = 0

for images, _ in train_no_norm_loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_samples += batch_samples

mean /= total_samples
std /= total_samples

# Apply transformations to dataset
flowersTransform = transforms.Compose([
    transforms.Resize((imageWidth, imageHeight)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

flowersTrain = Flowers102(root = "./data", split = "train", download=True, transform = flowersTransform)
flowersTest = Flowers102(root = "./data", split = "test", download=True, transform = flowersTransform)

# Move model onto GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create DataLoaders for batch usage
train_loader = DataLoader(flowersTrain, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(flowersTest, batch_size=batch_size)

imageChannels = 3 #R,G,B
imageSize = imageWidth*imageHeight*imageChannels


Initialise classifier, optimiser and loss function. Then train the model and output loss per epoch.

In [2]:
classifier = FlowersNetwork(in_feature = imageSize, hidden_layers = [32, 16, 8], out_features = 102, activation_function = F.relu)

# Move model to GPU
classifier.to(device)

lossFunction = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(classifier.parameters(), lr=0.0001)

classifier.train()
epochs = 500
patience = 3 
losses = []
for i in range(epochs):
  epochLoss = 0.0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    images = images.view(-1, imageSize)

    predictions = classifier.forward(images)

    loss = lossFunction(predictions, labels)
    epochLoss += loss.item()

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

  epochLoss /= len(train_loader)
  losses.append(epochLoss)
  print(f"Epoch {i+1} - {epochLoss}")

# Plot graph
plt.plot(range(len(losses)), losses)
plt.ylabel("Loss")
plt.xlabel("Epoch")

Epoch 1 - 4.676694840192795
Epoch 2 - 4.5916172713041306
Epoch 3 - 4.518186688423157
Epoch 4 - 4.425028428435326
Epoch 5 - 4.339201524853706
Epoch 6 - 4.256075501441956


KeyboardInterrupt: 

Calculate model accuracy

In [ ]:
with torch.no_grad():
  classifier.eval()
  correct = 0
  total = 0
  for images, labels in flowersTest:
        images = images.to(device)
        images = images.view(-1, imageSize)
        predictions = classifier.forward(images)
        _, predictedClass = torch.max(predictions, dim=1)

        total += 1
        if predictedClass.item() == labels:
            correct += 1
  accuracy = correct / total
  print(f"Model Accuracy: {accuracy}")

Predicted - 33
Actual - 0
Predicted - 61
Actual - 0
Predicted - 66
Actual - 0
Predicted - 0
Actual - 0
Predicted - 63
Actual - 0
Predicted - 69
Actual - 0
Predicted - 59
Actual - 0
Predicted - 0
Actual - 0
Predicted - 61
Actual - 0
Predicted - 79
Actual - 0
Predicted - 51
Actual - 0
Predicted - 12
Actual - 0
Predicted - 83
Actual - 0
Predicted - 61
Actual - 0
Predicted - 83
Actual - 0
Predicted - 18
Actual - 0
Predicted - 95
Actual - 0
Predicted - 35
Actual - 0
Predicted - 91
Actual - 0
Predicted - 39
Actual - 0
Predicted - 72
Actual - 1
Predicted - 12
Actual - 1
Predicted - 52
Actual - 1
Predicted - 68
Actual - 1
Predicted - 78
Actual - 1
Predicted - 5
Actual - 1
Predicted - 48
Actual - 1
Predicted - 93
Actual - 1
Predicted - 79
Actual - 1
Predicted - 101
Actual - 1
Predicted - 79
Actual - 1
Predicted - 79
Actual - 1
Predicted - 45
Actual - 1
Predicted - 56
Actual - 1
Predicted - 51
Actual - 1
Predicted - 99
Actual - 1
Predicted - 48
Actual - 1
Predicted - 79
Actual - 1
Predicted - 84

Model's accuracy (on my run): 3.15% - very low, only slightly better than randomly picking.
This was a decent first attempt at building a Neural Network in terms of grasping how the layout works.
A linear Neural Network however is not suitable for image classification, we will have to look into Convolutional Neural Networks going forward.